# DEH 30-Day PySpark Challenge
## Day 18 — UDFs: User Defined Functions

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
import hashlib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import udf
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

In [ ]:
orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name",  StringType(), True),
    StructField("last_name",   StringType(), True),
    StructField("email",       StringType(), True),
    StructField("city",        StringType(), True),
    StructField("state",       StringType(), True),
    StructField("country",     StringType(), True),
    StructField("signup_date", DateType(),   True),
    StructField("segment",     StringType(), True)
])

orders_df    = spark.read.option("header","true").schema(orders_schema).csv(f"s3a://{BUCKET}/data/orders.csv")
customers_df = spark.read.option("header","true").schema(customers_schema).csv(f"s3a://{BUCKET}/data/customers.csv")
print(f"Orders: {orders_df.count()} | Customers: {customers_df.count()}")

## Step 5 — Creating a UDF with F.udf()

In [ ]:
# Step 1 — define a Python function
def categorize_price(price):
    if price is None:
        return "Unknown"
    elif price > 500:
        return "Premium"
    elif price > 100:
        return "Standard"
    else:
        return "Budget"

# Step 2 — register as UDF with return type
categorize_price_udf = F.udf(categorize_price, StringType())

# Step 3 — apply to a column
orders_df.withColumn(
    "price_category",
    categorize_price_udf(F.col("unit_price"))
).select("order_id", "unit_price", "price_category").show(5)

## Step 6 — A second UDF example

In [ ]:
# Step 1 — define a Python function
def get_payment_type(method):
    if method is None:
        return "Unknown"
    elif "Card" in method:
        return "Card"
    else:
        return "Digital Wallet"

# Step 2 — register as UDF with return type
get_payment_type_udf = F.udf(get_payment_type, StringType())

# Step 3 — apply to a column
orders_df.withColumn(
    "payment_type",
    get_payment_type_udf(F.col("payment_method"))
).select("order_id", "payment_method", "payment_type").show(5)

## Step 7 — Native function equivalents (always prefer these)

In [ ]:
# Replace categorize_price UDF with when/otherwise
orders_df.withColumn(
    "price_category",
    F.when(F.col("unit_price") > 500,  "Premium")
     .when(F.col("unit_price") > 100,  "Standard")
     .otherwise("Budget")
).select("order_id", "unit_price", "price_category").show(5)

# Replace get_payment_type UDF with when/contains
orders_df.withColumn(
    "payment_type",
    F.when(F.col("payment_method").contains("Card"), "Card")
     .otherwise("Digital Wallet")
).select("order_id", "payment_method", "payment_type").show(5)

## Step 8 — When a UDF is justified: SHA256 hashing

In [ ]:
def hash_email(email):
    if email is None:
        return None
    return hashlib.sha256(email.encode()).hexdigest()

hash_email_udf = F.udf(hash_email, StringType())

customers_df.withColumn(
    "hashed_email",
    hash_email_udf(F.col("email"))
).select("customer_id", "email", "hashed_email").show(3, truncate=False)

## Step 9 — Pandas UDF: crossing the boundary once per partition

In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

# Step 1 — define a Python function
# Type hints are required — PySpark uses them to set up the Arrow schema
def categorize_price_pandas(price: pd.Series) -> pd.Series:
    return price.apply(
        lambda p: "Unknown" if p is None
        else "Premium"  if p > 500
        else "Standard" if p > 100
        else "Budget"
    )

# Step 2 — register as a Pandas UDF with return type
# Same pattern as F.udf() — entire partition crosses as an Arrow batch
categorize_price_pudf = pandas_udf(categorize_price_pandas, StringType())

# Step 3 — apply exactly like a regular UDF
orders_df.withColumn(
    "price_category",
    categorize_price_pudf(F.col("unit_price"))
).select("order_id", "unit_price", "price_category").show(5)

---
## Assignment — Day 18

---

### Task 1
Write a UDF called `classify_order` that takes `unit_price` and `quantity` and returns:  
- "High Value" if price × quantity > 1000  
- "Medium Value" if > 200  
- "Low Value" otherwise  

Apply it to `orders.csv`.

In [ ]:
# Task 1 — Write your code here


### Task 2
Rewrite Task 1 using only native Spark functions (`when()` / `otherwise()`).  
Compare the two approaches — which is cleaner?

In [ ]:
# Task 2 — Write your code here


### Task 3
Write a UDF that takes `first_name` and `last_name` and returns initials.  
Example: "James Anderson" → "J.A."  
Apply it to `customers.csv`.

In [ ]:
# Task 3 — Write your code here


### Task 4
Try to rewrite Task 3 using only native Spark functions.  
Can you do it? What does this tell you about when a UDF is justified?

In [ ]:
# Task 4 — Write your code here


### Task 5
Rewrite Task 1 (`classify_order`) as a Pandas UDF.  
Your function must accept a `pd.Series` and return a `pd.Series`.  
Register it with `pandas_udf()` — no decorator.  
Apply it to `orders.csv` and verify the output matches Task 1.

In [ ]:
# Task 5 — Write your code here


### Task 6
Write a Pandas UDF called `flag_high_discount` that takes `discount_pct` as a `pd.Series`  
and returns `"Yes"` if the discount is 15% or more, `"No"` otherwise.  
Apply it to `orders.csv` and show `order_id`, `discount_pct`, and `high_discount` for all rows where `high_discount = "Yes"`.

In [ ]:
# Task 6 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*